# Kansas — Chapter 40 (Insurance) → `data/kansas/ins_codes/*.md`

Kansas’s **insurance code** is **Chapter 40** of the **Kansas Statutes** (official hub: [ksrevisor.org](https://www.ksrevisor.org/statutes2024/)). This notebook mirrors **Justia’s** browse tree: **[Chapter 40 — Insurance](https://law.justia.com/codes/kansas/chapter-40/)** (`/codes/kansas/chapter-40/…`).

Justia nests **article** pages, then **`section-40-…`** URLs (for example [`…/section-40-101/`](https://law.justia.com/codes/kansas/chapter-40/article-1/section-40-101/)). **Cloudflare** often blocks plain **`httpx`**; we use **`curl_cffi`** with **`impersonate="chrome120"`** (same pattern as **`ins_ipynb/idaho.ipynb`** / **`georgia.ipynb`**).

**Discovery:** BFS from the Chapter 40 index, following paths under **`/codes/kansas/chapter-40/`** that are not **`section-40-…`** pages; every **`section-40-…`** link is collected (~**1,400** sections).

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`KSA_sec_<label>.md`** where **`label`** is the URL tail after **`section-`** with hyphens replaced by underscores (e.g. `KSA_sec_40_101.md`).

**Config:** Discovery starts at the stable canonical **`/codes/kansas/chapter-40/`** (year-scoped **`/codes/kansas/{year}/…`** URLs may 404 until Justia publishes that edition). **`MAX_SECTIONS`** caps **downloads** (**0** = all). **`MAX_DISCOVERY_PAGES`** caps **index** fetches during discovery (**0** = no cap). **`REUSE_DISCOVERED_URLS`** skips a repeat crawl when **`_kansas_chapter40_section_urls.txt`** exists.

**Politeness:** **`REQUEST_DELAY_SEC`** between requests.

Then run **`python -m app.ingest`** from the project root.


In [ ]:
%pip install -q curl_cffi beautifulsoup4


In [ ]:
from __future__ import annotations

import re
import time
from collections import deque
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
PATH_PREFIX = "/codes/kansas/chapter-40"
TITLE_INDEX = f"{BASE}/codes/kansas/chapter-40/"

OUT_DIR = Path("data") / "kansas" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_SECTIONS = 0
MAX_DISCOVERY_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_kansas_chapter40_section_urls.txt"
REUSE_DISCOVERED_URLS = True

section_label_re = re.compile(r"/section-(40[^/]+)/?$", re.I)


In [6]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def discover_section_urls() -> list[str]:
    """BFS article index pages; collect section-40-… URLs."""
    start = TITLE_INDEX
    seen: set[str] = set()
    in_q: set[str] = {path_key(start)}
    q: deque[str] = deque([start])
    sections: set[str] = set()
    fetches = 0
    while q:
        if MAX_DISCOVERY_PAGES and fetches >= MAX_DISCOVERY_PAGES:
            break
        url = q.popleft()
        pk = path_key(url)
        in_q.discard(pk)
        if pk in seen:
            continue
        if "/section-40-" in pk.lower():
            continue
        seen.add(pk)
        html = curl_get(url)
        fetches += 1
        soup = BeautifulSoup(html, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(url, a["href"])
            p = path_key(absu)
            if not p.startswith(PATH_PREFIX):
                continue
            if "/section-40-" in p.lower():
                sections.add(BASE + p + "/")
            else:
                if "/section-" in p.lower() and "/section-40-" not in p.lower():
                    continue
                if p in seen or p in in_q:
                    continue
                in_q.add(p)
                q.append(BASE + p + "/")
    return sorted(sections, key=lambda u: label_sort_key(section_label_from_url(u)))


def section_label_from_url(url: str) -> str:
    path = urlparse(url).path.rstrip("/")
    m = section_label_re.search(path)
    if not m:
        raise ValueError(f"cannot parse section id from {url!r}")
    return m.group(1)


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"KSA_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and "Kansas Stat" in s:
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        if s.startswith("KS Stat §"):
            continue
        if s.startswith("Disclaimer:") or s.startswith("These codes may not"):
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_chapter40() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(section_label_from_url(u)))
        print(f"Loaded {len(all_urls)} section URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_section_urls()
        print(f"Discovered {len(found)} section URLs under Chapter 40")
        all_urls = found
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_SECTIONS else all_urls[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited downloads to first {len(todo)} sections (MAX_SECTIONS)")

    wrote = skipped = failed = 0
    for i, sec_url in enumerate(todo, 1):
        label = section_label_from_url(sec_url)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(sec_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t.split("::", 1)[0].strip() if head_t else f"Kansas Statutes § {label}"
                md = (
                    f"# {title}\n\n"
                    f"**Kansas Statutes — Chapter 40 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {sec_url}\n\n"
                    f"**Verify on official site:** [ksrevisor.org](https://www.ksrevisor.org/statutes2024/)\n\n"
                    f"**Section:** §{label}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_chapter40()


Exception ignored from cffi callback <function buffer_callback at 0x11152eaf0>:
Traceback (most recent call last):
  File "/Users/apps/Downloads/ZProjects/RAG/.venv/lib/python3.9/site-packages/curl_cffi/curl.py", line 100, in buffer_callback
    @ffi.def_extern()
KeyboardInterrupt: 


FAIL 40-2-103: Failed to perform, curl: (23) Failure writing output to destination, passed 13 returned 0. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
… 200/1436 (wrote=199 skipped=0 failed=1)
… 400/1436 (wrote=399 skipped=0 failed=1)
… 600/1436 (wrote=599 skipped=0 failed=1)
… 800/1436 (wrote=799 skipped=0 failed=1)
… 1000/1436 (wrote=999 skipped=0 failed=1)
… 1200/1436 (wrote=1199 skipped=0 failed=1)
… 1400/1436 (wrote=1399 skipped=0 failed=1)
Done. wrote=1435 skipped=0 failed=1 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/kansas/ins_codes


{'wrote': 1435, 'skipped': 0, 'failed': 1}

## Next step

`python -m app.ingest` from the project root.
